
# 1.3.5.4 Concordancia de patrones estructurado (*Structural Pattern Matching*) — `match` / `case`

**Objetivo:** Dominar `match-case` de Python (>=3.10) para **desestructurar datos** (literales, secuencias, mapeos, clases) y decidir en función de su **forma y contenido**, con **guardas** y **variantes** aplicadas a dominios de **IA** y **Ciencia de Datos**.



## Índice
1. [Conceptos clave](#conceptos)
2. [Patrones y sintaxis](#patrones)
    - [Patrones literales y comodín `_`](#literales)
    - [Patrón OR `|`](#or)
    - [Captura y comodines (`name`, `_`, `as`)](#captura)
    - [Guardas con `if` en `case`](#guardas)
    - [Secuencias (`list`, `tuple`, slicing con `*`)](#secuencias)
    - [Mapeos (`dict`)](#mapeos)
    - [Clases/`dataclass`/`NamedTuple` y `__match_args__`](#clases)
    - [Envoltorios comunes: `Enum`, `typing`](#enum)
3. [Orden y alcance de casos: mejores prácticas](#orden)
4. [Errores comunes y sutilezas](#errores)
5. [Ejercicios resueltos por secciones](#ejercicios)
6. [Reto integrador (IA & Datos)](#reto)



## <a id="conceptos"></a> 1) Conceptos clave

- `match expr:` compara **`expr`** contra **patrones** en orden top–down.
- Cada `case` tiene un **patrón** y opcionalmente una **guarda** (`if condición`).  
- Los patrones no son expresiones arbitrarias: describen **formas/estructuras**.
- Coincidencia **estructural**: no solo valores, también la forma del dato (longitud, llaves/atributos, etc.).
- Si varios patrones podrían aplicar, **gana el primero** que coincida.
- Internamente, se apoya en protocolos como `__match_args__` (para clases).



## <a id="patrones"></a> 2) Patrones y sintaxis



### <a id="literales"></a> 2.1 Patrones literales y comodín `_`

- Literales: `0`, `1.0`, `"ok"`, `True`, `None`.
- Comodín `_`: coincide con **cualquier cosa** (no captura).


In [1]:

def http_status_desc(code):
    match code:
        case 200:
            return "OK"
        case 201:
            return "Created"
        case 400 | 404:
            return "Bad Request / Not Found"
        case 500:
            return "Internal Server Error"
        case _:
            return "Unknown"

[http_status_desc(c) for c in [200, 201, 400, 404, 418]]


['OK',
 'Created',
 'Bad Request / Not Found',
 'Bad Request / Not Found',
 'Unknown']


### <a id="or"></a> 2.2 Patrón OR `|`

Permite agrupar alternativas con la **misma acción**.


In [2]:

def es_vocal(ch: str) -> bool:
    match ch.lower():
        case "a" | "e" | "i" | "o" | "u":
            return True
        case _:
            return False

[es_vocal(c) for c in "aeioux"]


[True, True, True, True, True, False]


### <a id="captura"></a> 2.3 Captura y comodines (`name`, `_`, `as`)

- **Captura**: `case x:` captura el valor en `x` (¡ojo con colisiones de nombres!).  
- **`_`**: comodín sin captura (convención: no usado después).  
- **`as`**: captura el **todo** tras validar subpatrones, p. ej. `[x, y] as punto`.


In [3]:

data = [3, 4]
match data:
    case [x, y] as punto:
        print("x=", x, "y=", y, "| punto=", punto)
    case _:
        print("otro")


x= 3 y= 4 | punto= [3, 4]



### <a id="guardas"></a> 2.4 Guardas con `if` en `case`

Permiten **refinar** la coincidencia con una condición booleana arbitraria.


In [4]:

def clasifica_tupla(t):
    match t:
        case (x, y) if isinstance(x, int) and isinstance(y, int) and x <= y:
            return "par ordenado ascendente"
        case (x, y) if isinstance(x, int) and isinstance(y, int):
            return "par desordenado"
        case _:
            return "otra cosa"

[clasifica_tupla(v) for v in [(1,2),(5,3),("a",1)]]


['par ordenado ascendente', 'par desordenado', 'otra cosa']


### <a id="secuencias"></a> 2.5 Secuencias (`list`, `tuple`, slicing con `*`)

- Longitud fija: `[a, b]`.
- Longitud variable: `[primero, *medio, ultimo]`.
- Coincide por **tipo de protocolo de secuencia**, no solo `list`.


In [5]:

def describe_vector(v):
    match v:
        case []:
            return "vacío"
        case [x]:
            return f"unario ({x})"
        case [x, y]:
            return f"binario ({x},{y})"
        case [x, y, *resto]:
            return f"≥3 elementos: primeros=({x},{y}), resto={resto}"
        case _:
            return "no secuencia"

[describe_vector(v) for v in ([], [9], [1,2], [1,2,3,4], "abc")]


['vacío',
 'unario (9)',
 'binario (1,2)',
 '≥3 elementos: primeros=(1,2), resto=[3, 4]',
 'no secuencia']


### <a id="mapeos"></a> 2.6 Mapeos (`dict`)

- Selección por **llaves** y captura de valores: `{"k": v}`.
- Patrones parciales: pueden faltar otras llaves y seguir coincidiendo.


In [6]:

def valida_payload(p):
    match p:
        case {"id": int(i), "name": str(nombre)}:  # validación de tipos con captura
            return ("ok", i, nombre)
        case {"id": int(i)}:
            return ("faltan_campos", i)
        case _:
            return ("invalido", None)

tests = [
    {"id": 7, "name": "Ada", "role": "admin"},
    {"id": 3},
    {"id": "x"},
]
[valida_payload(t) for t in tests]


[('ok', 7, 'Ada'), ('faltan_campos', 3), ('invalido', None)]


### <a id="clases"></a> 2.7 Clases, `dataclass`, `NamedTuple` y `__match_args__`

- Coincidencia por **tipo** y **atributos posicionales** definidos en `__match_args__`.
- `dataclass` genera `__match_args__` automáticamente con sus campos.
- `NamedTuple` y `typing.NamedTuple` también soportan coincidencia posicional.


In [7]:

from dataclasses import dataclass
from typing import NamedTuple

@dataclass
class Punto:
    x: int
    y: int

class Medida(NamedTuple):
    valor: float
    unidad: str

def procesa(obj):
    match obj:
        case Punto(0, 0):
            return "origen"
        case Punto(x, y) if x == y:
            return f"sobre y=x (x=y={x})"
        case Punto(x, y):
            return f"punto general ({x},{y})"
        case Medida(v, "ms"):
            return f"{v} milisegundos"
        case Medida(v, u):
            return f"{v} {u}"
        case _:
            return "desconocido"

[procesa(o) for o in (Punto(0,0), Punto(3,3), Punto(1,2), Medida(12.5,"ms"), Medida(1.2,"kg"))]


['origen',
 'sobre y=x (x=y=3)',
 'punto general (1,2)',
 '12.5 milisegundos',
 '1.2 kg']


### <a id="enum"></a> 2.8 Enums y envoltorios comunes

Útil para **estados** o **eventos**.


In [8]:

from enum import Enum, auto

class Estado(Enum):
    INICIAL = auto()
    ENTRENANDO = auto()
    EVALUANDO = auto()
    FINALIZADO = auto()

def siguiente(e: Estado):
    match e:
        case Estado.INICIAL:
            return Estado.ENTRENANDO
        case Estado.ENTRENANDO:
            return Estado.EVALUANDO
        case Estado.EVALUANDO:
            return Estado.FINALIZADO
        case _:
            return e

[siguiente(x) for x in Estado]


[<Estado.ENTRENANDO: 2>,
 <Estado.EVALUANDO: 3>,
 <Estado.FINALIZADO: 4>,
 <Estado.FINALIZADO: 4>]


## <a id="orden"></a> 3) Orden y alcance: mejores prácticas

- Ordena los `case` de **más específico** a **más general**.
- Evita capturas demasiado generales antes que patrones precisos.
- Usa **guardas** para refinar sin duplicar patrones.
- Prefiere `match` cuando la **forma** del dato guía la decisión; usa `if/elif` si solo hay simples booleanos.



## <a id="errores"></a> 4) Errores comunes y sutilezas

- Confundir **captura** con comparación: `case x:` **captura**; para comparar usa literales, enums o `case x if x == valor:`.
- Con `dict`, el patrón **no exige** exclusividad de llaves: si faltan las requeridas no coincide; si sobran, **sí** coincide.
- La comprobación de tipo en mapeos con `int()`/`str()` en el patrón aplica conversión **estructural**; úsalo para validar.
- No abuses de `match` si un `if/elif` simple es más claro.



## <a id="ejercicios"></a> 5) Ejercicios resueltos por secciones
A continuación, ejercicios con **variantes** y **soluciones**.



### 5.1 Literales y `|` (OR)
**Tarea:** Dado un código de severidad, clasifícalo.
- `0` → "OK"
- `1 | 2` → "ADVERTENCIA"
- `3 | 4 | 5` → "ERROR"
- otro → "DESCONOCIDO"



### 5.2 Captura y `as` + guardas
**Tarea:** Dado un string, si es no vacío y todo minúsculas, devolver `"lower:<s>"`;  
si no vacío y todo mayúsculas, `"UPPER:<s>"`; en otro caso `"mixto:<s>"`; si vacío `"vacio"`.


In [9]:

def forma_texto(s: str):
    match s:
        case "" | "   ":
            return "vacio"
        case str(t) if t.islower():
            return f"lower:{t}"
        case str(t) if t.isupper():
            return f"UPPER:{t}"
        case str(t) as completo:
            return f"mixto:{completo}"
        case _:
            return "no-texto"

[forma_texto(x) for x in ["", "abc", "ABC", "AaBb", 123]]


['vacio', 'lower:abc', 'UPPER:ABC', 'mixto:AaBb', 'no-texto']


### 5.3 Secuencias y slicing con `*`
**Tarea:** Dado un vector, devuelve:
- `"origin"` si `[0, 0]`
- `"axis-x"` si `[x, 0]` con `x != 0`
- `"axis-y"` si `[0, y]` con `y != 0`
- `"longitud>2"` si hay 3 o más elementos
- `"otro"` en caso contrario


In [10]:

def clasifica_vec(v):
    match v:
        case [0, 0]:
            return "origin"
        case [x, 0] if x != 0:
            return "axis-x"
        case [0, y] if y != 0:
            return "axis-y"
        case [_, _, *rest]:
            return "longitud>2"
        case _:
            return "otro"

[clasifica_vec(v) for v in ([0,0],[3,0],[0,-2],[1,2,3],[7])]


['origin', 'axis-x', 'axis-y', 'longitud>2', 'otro']


### 5.4 Mapeos (dict) con validación de tipos
**Tarea:** Recibir eventos de logging con forma `{"lv": <str>, "msg": <str>, ...}` y normalizar el nivel a `DEBUG/INFO/WARN/ERROR`.


In [11]:

def normaliza_evento(e):
    match e:
        case {"lv": str(lv), "msg": str(m)} if lv.lower() in {"debug","info","warn","warning","error"}:
            base = lv.upper().replace("WARNING","WARN")
            return {"lv": base, "msg": m, **{k:v for k,v in e.items() if k not in {"lv","msg"}}}
        case {"msg": str(m)}:
            return {"lv": "INFO", "msg": m}
        case _:
            return {"lv": "INFO", "msg": str(e)}

[normaliza_evento(x) for x in [
    {"lv":"debug","msg":"init"},
    {"lv":"warning","msg":"disk"},
    {"msg":"sin nivel"},
    123
]]


[{'lv': 'DEBUG', 'msg': 'init'},
 {'lv': 'WARN', 'msg': 'disk'},
 {'lv': 'INFO', 'msg': 'sin nivel'},
 {'lv': 'INFO', 'msg': '123'}]


### 5.5 Clases/`dataclass`/`NamedTuple`
**Tarea:** Distinguir puntos, medidas y vectores dispersos.


In [12]:

from typing import NamedTuple

@dataclass
class Vector:
    x: float
    y: float
    z: float = 0.0

class SparseVec(NamedTuple):
    idx: list[int]
    val: list[float]

def describe(o):
    match o:
        case Vector(0, 0, 0):
            return "vector nulo"
        case Vector(x, y, z) if abs(x)==abs(y)==abs(z):
            return "equilibrado"
        case Vector(x, y, z):
            return f"vector({x},{y},{z})"
        case SparseVec([int() as idx1, *rest_i], [float() as v1, *rest_v]):
            return f"sparse con {1+len(rest_i)} entradas"
        case _:
            return "otro"

[describe(x) for x in [Vector(0,0,0), Vector(1,-1,1), Vector(1,0,2), SparseVec([0,3],[1.0,2.0])]]


['vector nulo', 'equilibrado', 'vector(1,0,2)', 'sparse con 2 entradas']


## <a id="reto"></a> 6) Reto integrador (IA & Datos)

**Problema:** Implementa un **router de tareas** para un mini-pipeline de datos/IA.  
Las entradas (`evento`) pueden ser:
- Tupla `("read_csv", ruta)`
- Tupla `("scale", vector)` donde `vector` sea lista/tupla de números
- Diccionario `{"train": {"X": <matriz-like>, "y": <lista>}}`
- Diccionario `{"predict": {"X": <matriz-like>}}`
- Texto con prefijos: `"help"`, `"status?"`, o cualquier otro comando
- En caso desconocido, devolver `("error","evento no reconocido")`

**Requisitos:**
- Usa **`match-case`** con **guardas** para validar tipos y formas.
- Retorna tuplas **(acción, payload_normalizado)** listas para consumir por otro módulo.


In [13]:

def es_vector_num(v):
    try:
        return isinstance(v, (list, tuple)) and v and all(isinstance(x, (int,float)) for x in v)
    except Exception:
        return False

def es_matriz_num(M):
    try:
        return isinstance(M, (list, tuple)) and M and all(isinstance(f, (list, tuple)) and f and all(isinstance(x,(int,float)) for x in f) for f in M)
    except Exception:
        return False

def router(evento):
    match evento:
        case ("read_csv", str(ruta)):
            return ("read_csv", {"path": ruta})
        case ("scale", v) if es_vector_num(v):
            # Normalizado min-max
            lo, hi = min(v), max(v)
            if lo == hi:
                return ("scale", {"vector":[0.0 for _ in v]})
            return ("scale", {"vector":[(x-lo)/(hi-lo) for x in v]})
        case {"train": {"X": X, "y": y}} if es_matriz_num(X) and isinstance(y, (list,tuple)) and len(y)==len(X):
            return ("train", {"X": X, "y": list(y)})
        case {"predict": {"X": X}} if es_matriz_num(X):
            return ("predict", {"X": X})
        case str(cmd) if cmd.strip().lower() == "help":
            return ("help", {})
        case str(cmd) if cmd.strip().endswith("?"):
            return ("query", {"text": cmd.strip()})
        case _:
            return ("error", "evento no reconocido")

pruebas = [
    ("read_csv","/tmp/data.csv"),
    ("scale",[10, 20, 30]),
    {"train":{"X":[[1,2],[3,4]], "y":[0,1]}},
    {"predict":{"X":[[0.1,0.2],[0.3,0.4]]}},
    "status?",
    123,
]
[router(ev) for ev in pruebas]


[('read_csv', {'path': '/tmp/data.csv'}),
 ('scale', {'vector': [0.0, 0.5, 1.0]}),
 ('train', {'X': [[1, 2], [3, 4]], 'y': [0, 1]}),
 ('predict', {'X': [[0.1, 0.2], [0.3, 0.4]]}),
 ('query', {'text': 'status?'}),
 ('error', 'evento no reconocido')]